In [4]:
import pandas as pd
import numpy as np
import joblib, time, json, hashlib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
from datasketch import MinHash, MinHashLSH

class ScamStopEngine:
    def __init__(self, lsh_threshold=0.9, num_perm=128):
        self.vectorizer    = TfidfVectorizer(ngram_range=(1, 2))
        self.classifier    = MultinomialNB()
        self.lsh           = MinHashLSH(threshold=lsh_threshold, num_perm=num_perm)
        self.lsh_minhashes = {}   # key → MinHash — used to compute actual Jaccard similarity
        self.num_perm      = num_perm
        self.lsh_threshold = lsh_threshold
        self.b = getattr(self.lsh, '_b', 20)
        self.r = getattr(self.lsh, '_r', 4)
        self.performance_data = None

    def _get_minhash(self, text, n=3):
        """
        Generates a MinHash signature based on character-level n-grams
        to ensure structural resiliency against adversarial typos.
        """
        m = MinHash(num_perm=self.num_perm)

        # Normalize text to lowercase and remove extraneous whitespace
        clean_text = " ".join(str(text).lower().split())

        # Generate character-level n-grams (default 3-grams)
        for i in range(len(clean_text) - n + 1):
            shingle = clean_text[i:i+n]
            m.update(shingle.encode('utf8'))

        return m

    def _get_bands(self, m):
        """Yield (band_idx, band_hash) pairs — matches server.py exactly."""
        v = m.hashvalues
        for i in range(self.b):
            band = v[i * self.r: (i + 1) * self.r]
            yield (i, hashlib.sha1(str(list(band)).encode('utf-8')).hexdigest())

    def train_and_evaluate(self, scam_csv, safe_csv):
        print('[TRAIN] Loading datasets...')
        scam_df = pd.read_csv(scam_csv, engine='python', on_bad_lines='skip', encoding='ISO-8859-1')
        safe_df = pd.read_csv(safe_csv, engine='python', on_bad_lines='skip', encoding='ISO-8859-1')
        scam_df['label'] = 1
        safe_df['label'] = 0
        df = pd.concat([scam_df[['text', 'label']], safe_df[['text', 'label']]])
        df = df.dropna(subset=['text'])
        df['text'] = df['text'].astype(str).str.strip()

        # Explicit String Deduplication Phase
        df = df.drop_duplicates(subset='text')
        print(f"[TRAIN] {len(df)} rows ({df['label'].sum()} scam, {(df['label']==0).sum()} safe)")

        # Stratified Train-Test Split (80/20)
        X_train, X_test, y_train, y_test = train_test_split(
            df['text'], df['label'], test_size=0.20, random_state=42, stratify=df['label']
        )

        # Rebuild LSH index from scratch to clear out stale signatures
        print('[TRAIN] Rebuilding LSH index (Blacklist Cache)...')
        self.lsh = MinHashLSH(threshold=self.lsh_threshold, num_perm=self.num_perm)
        self.b   = getattr(self.lsh, '_b', 20)
        self.r   = getattr(self.lsh, '_r', 4)
        scam_train_texts = []

        # Core Architecture: Only insert known SCAMS into the structural LSH Tier 1 index
        for idx, text in X_train[y_train == 1].items():
            try:
                mh = self._get_minhash(text)
                self.lsh.insert(f'scam_{idx}', mh)
                self.lsh_minhashes[f'scam_{idx}'] = mh
                scam_train_texts.append(text)
            except Exception:
                pass

        print('[TRAIN] Training TF-IDF Semantic Feature Extractor + Multinomial Naive Bayes...')
        X_tr = self.vectorizer.fit_transform(X_train.astype(str))
        self.classifier.fit(X_tr, y_train)

        print('[TRAIN] Evaluating hybrid pipeline on test set...')
        y_pred, y_proba, lats = [], [], []
        tier_hits = {'LSH': 0, 'NLP': 0}
        for msg in X_test.astype(str):
            t0 = time.time()
            mh = self._get_minhash(msg)
            try:
                hits = self.lsh.query(mh)
            except Exception:
                hits = []

            # Tier 1 Short-Circuit Logic Optimization Block
            if hits:
                y_pred.append(1)
                y_proba.append(0.99)
                tier_hits['LSH'] += 1
            else:
                # Tier 2 Semantic Routing Processing Block
                p = self.classifier.predict_proba(self.vectorizer.transform([msg]))[0][1]
                y_proba.append(float(p))
                # Optimized decision classification threshold at t = 0.7
                y_pred.append(1 if p > 0.7 else 0)
                tier_hits['NLP'] += 1

            # Differential timestamp tracking to measure exact pipeline latency
            lats.append((time.time() - t0) * 1000)

        y_pred  = np.array(y_pred)
        y_proba = np.array(y_proba)
        cm  = confusion_matrix(y_test, y_pred).tolist()
        cr  = classification_report(y_test, y_pred, output_dict=True)
        roc = round(float(roc_auc_score(y_test, y_proba)), 4) if len(np.unique(y_test)) > 1 else None
        avg_ms = round(float(np.mean(lats)), 2)
        try:
            bands = int(self.lsh._b); rows = int(self.lsh._r)
        except AttributeError:
            bands = self.b; rows = self.r

        # performance_data schema mapping directly to Table 4 and Table 5
        self.performance_data = {
            'performance_metrics': {
                'accuracy':  round(float(accuracy_score(y_test, y_pred)), 4),
                'precision': round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
                'recall':    round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
                'f1_score':  round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
                'auc_roc':   roc,
                'lsh_similarity_threshold': self.lsh_threshold,
            },
            'confusion_matrix': {
                'true_negative':  cm[0][0], 'false_positive': cm[0][1],
                'false_negative': cm[1][0], 'true_positive':  cm[1][1],
            },
            'lsh_configurations': {
                'hash_functions_k':      self.num_perm,
                'bands_b':               bands,
                'rows_per_band_r':       rows,
                'lsh_threshold':         self.lsh_threshold,
                'minhash_shingle_size':  'Word-based (3-gram)',
                'vocabulary_size_tfidf': len(self.vectorizer.vocabulary_),
                'avg_query_time_ms':     avg_ms,
            },
            'classification_report': cr,
        }
        pm = self.performance_data['performance_metrics']
        print(f"\\n[RESULTS] Accuracy={pm['accuracy']*100:.2f}%  Precision={pm['precision']*100:.2f}%  Recall={pm['recall']*100:.2f}%  F1={pm['f1_score']*100:.2f}%  AUC={pm['auc_roc']}")
        print(f"  LSH hits: {tier_hits['LSH']}  NLP hits: {tier_hits['NLP']}  Avg latency: {avg_ms} ms")
        return X_test, y_test, scam_train_texts

    def predict(self, message):
        mh = self._get_minhash(message)
        try:
            hits = self.lsh.query(mh)
            if hits:
                best_sim = max(
                    (mh.jaccard(self.lsh_minhashes[k]) for k in hits if k in self.lsh_minhashes),
                    default=self.lsh_threshold
                )
                return f'SCAM (Detected via LSH Near-Duplicate, Similarity: {best_sim*100:.2f}%)'
        except Exception:
            pass
        p = self.classifier.predict_proba(self.vectorizer.transform([message]))[0][1]
        if p > 0.7:
            return f'SCAM (Detected via NLP Analysis, Confidence: {p*100:.2f}%)'
        return 'SAFE'


# ---------------------------------------------------------------------------
# Training Execution Target
# ---------------------------------------------------------------------------
if __name__ == '__main__':
    engine = ScamStopEngine(lsh_threshold=0.9, num_perm=128)
    X_test_data, y_test_data, scam_train_texts = engine.train_and_evaluate('scam_messages.csv', 'safe_messages.csv')

    # Save to binary file
    joblib.dump(engine, 'scam_stop_engine.joblib')
    print(f'\\nSaved scam_stop_engine.joblib  b={engine.b}  r={engine.r}  threshold={engine.lsh_threshold}')

\nSaved scam_stop_engine.joblib  b=20  r=4  threshold=0.9
